<a href="https://colab.research.google.com/github/VeriAnalisti-VA/desktop-tutorial/blob/main/1_Figen_Olist_Veri_Analizi_Projesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Olist Veri Analizi Projesi**

1.Görev: Veri Yükleme, İnceleme ve Temizleme

In [5]:
#Kütüphaneler
import pandas as pd
import numpy as np
import plotly.express as px

In [6]:
#Drive bağlantı izni
from google.colab import drive
drive.mount ('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
#Drive (Klasör) kısayol tanımlama
path = '/content/drive/MyDrive/Workintech Python Sunum Projesi/'

In [8]:
#Data Set dosyalarını yükleme
orders=pd.read_csv(path+'olist_orders_dataset.csv')
sellers=pd.read_csv(path+'olist_sellers_dataset.csv')
products=pd.read_csv(path+'olist_products_dataset.csv')
reviews=pd.read_csv(path+'olist_order_reviews_dataset.csv')
payments=pd.read_csv(path+'olist_order_payments_dataset.csv')
items=pd.read_csv(path+'olist_order_items_dataset.csv')
customers=pd.read_csv(path+'olist_customers_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Workintech Python Sunum Projesi/olist_orders_dataset.csv'

In [ ]:
#Tabloları görüntüleme -()- ilk 5 satır döner. Değer old. zaman, değer kadar satır döner.
orders.head()

In [ ]:
sellers.head()

In [ ]:
products.head()

In [ ]:
reviews.head()

In [ ]:
payments.head()

In [ ]:
items.head()

In [ ]:
customers.head()

In [ ]:
#Satır-Sütun Sayısı ve Sütun Adlarını Listeleme
for name, df in [('orders', orders), ('customers', customers), ('products', products),
                  ('sellers', sellers), ('items', items), ('payments', payments), ('reviews', reviews)]:
  print(f'\n=== {name} ===') #Tablo başlık adı ekleme
  print(f'Boyut: {df.shape}')

  #Veri Tipi ve boş hücre sayısı
  print(f'Veri Tipleri:\n{df.dtypes}')
  print(f'Eksik değerler:\n{df.isnull().sum()}')

In [ ]:
# Veri Tipi (datetime) formatına dönüştürme
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Değişikliklerin kontrol çıktısı
orders.info()


In [ ]:
#Eksik Değerlerin nedenini belirleme
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

In [ ]:
#Eksik Değerlerin nedenini belirleme
products[products['product_category_name'].isnull()].head()

2.Görev: Tanımlayıcı İstatistikleri Hesapla

In [ ]:
#Temel İstatistikleri Hesaplama
for name, df in [('orders', orders), ('customers', customers), ('products', products),
                  ('sellers', sellers), ('items', items), ('payments', payments), ('reviews', reviews)]:
  print(f'\n=== {name} ===')
  print(df.describe()) #sayısal sütunlar için otomatik olarak count, mean, std, min, max ve medyanı hesaplar.

In [ ]:
#Ortalama sipariş fiyatında iptal siparişleri filtreleme
delivered_orders = orders[orders['order_status'] == 'delivered']
delivered_items = delivered_orders.merge(items, on='order_id')

#Ortalama sipariş fiyatı
print(f"Ortalama fiyat: {delivered_items['price'].mean():.2f}")

#Medyan sipariş fiyatı
print(f"Medyan fiyat: {delivered_items['price'].median():.2f}")

#Sipariş fiyatı standart sapması
print(f"Standart sapma: {delivered_items['price'].std():.2f}")

In [ ]:
#Sipariş fiyatı dağılımı histogramı
import plotly.express as px

fig = px.histogram(delivered_items,
                   x='price',
                   title='Sipariş Fiyatı Dağılımı',
                   labels={'price': 'Fiyat', 'count': 'Sipariş Sayısı'},
                   nbins=50)
fig.show()

In [ ]:
#Ürün Kategorisi Dağılımı
category_dist = delivered_items.merge(products[['product_id', 'product_category_name']], on='product_id')
category_counts = category_dist['product_category_name'].value_counts().head(10).reset_index()
category_counts.columns = ['kategori', 'siparis_sayisi']

fig = px.bar(category_counts,
             x='kategori',
             y='siparis_sayisi',
             title='En Çok Sipariş Edilen 10 Ürün Kategorisi')
fig.show()

In [ ]:
#Ödeme Yöntemi Dağılımı
delivered_payments = delivered_orders.merge(payments, on='order_id')

payment_counts = delivered_payments['payment_type'].value_counts().reset_index()
payment_counts.columns = ['odeme_yontemi', 'siparis_sayisi']

fig = px.pie(payment_counts,
             values='siparis_sayisi',
             names='odeme_yontemi',
             title='Ödeme Yöntemi Dağılımı')
fig.show()

In [ ]:
#Kargo Süresi İstatistikleri
delivered_orders_copy = delivered_orders.copy()
delivered_orders_copy['kargo_suresi'] = (delivered_orders_copy['order_delivered_customer_date'] -
                                          delivered_orders_copy['order_purchase_timestamp']).dt.days

print(f"Ortalama kargo süresi: {delivered_orders_copy['kargo_suresi'].mean():.1f} gün")
print(f"Medyan kargo süresi: {delivered_orders_copy['kargo_suresi'].median():.1f} gün")
print(f"Standart sapma: {delivered_orders_copy['kargo_suresi'].std():.1f} gün")

In [ ]:
#Kargo süresi kutu grafiği
fig = px.box(delivered_orders_copy,
             y='kargo_suresi',
             title='Kargo Süresi Dağılımı',
             labels={'kargo_suresi': 'Kargo Süresi (gün)'})

fig.show()

In [ ]:
#Müşteri Memnuniyeti Puanları
delivered_reviews = delivered_orders.merge(reviews, on='order_id')

print(f"Ortalama puan: {delivered_reviews['review_score'].mean():.2f}")
print(f"Medyan puan: {delivered_reviews['review_score'].median():.2f}")

memnun = (delivered_reviews['review_score'] >= 4).sum()
memnun_degil = (delivered_reviews['review_score'] < 4).sum()
toplam = len(delivered_reviews)

print(f"Memnun müşteri (4+): {memnun} ({memnun/toplam*100:.1f}%)")
print(f"Memnun olmayan (4 altı): {memnun_degil} ({memnun_degil/toplam*100:.1f}%)")

3.Görev: Korelasyonları İncele

In [ ]:
#Sipariş Fiyatı vs. Teslimat Süresi: Daha fazla harcama yapan müşterilerin teslimat sürelerine toleransı farklı mı?
df_analiz = delivered_orders_copy.merge(delivered_items[['order_id','price','product_id']], on='order_id') \
                                  .merge(delivered_reviews[['order_id','review_score']], on='order_id')

print(df_analiz[['price','kargo_suresi']].corr())

In [ ]:
#Sipariş Fiyatı vs. Teslimat Süresi grafği
fig = px.scatter(df_analiz,
                 x='price',
                 y='kargo_suresi',
                 title='Sipariş Fiyatı vs Teslimat Süresi',
                 labels={'price': 'Fiyat (BRL)', 'kargo_suresi': 'Teslimat Süresi (gün)'},
                 opacity=0.3,)

fig.show()

In [ ]:
#Sipariş Fiyatı vs. Memnuniyet: Harcama miktarı müşteri memnuniyetini etkiliyor mu?
print(df_analiz[['price','review_score']].corr())

In [ ]:
#Sipariş Fiyatı vs. Memnuniyet grafiği
fig = px.scatter(df_analiz,
                 x='price',
                 y='review_score',
                 title='Sipariş Fiyatı vs Müşteri Memnuniyeti',
                 labels={'price': 'Fiyat (BRL)', 'review_score': 'Memnuniyet Puanı'},
                 opacity=0.3,)

fig.show()

In [ ]:
#Ürün Kategorisi vs. Memnuniyet: Farklı kategorilerde memnuniyet seviyeleri değişiyor mu?
df_kategori = df_analiz.merge(products[['product_id','product_category_name']], on='product_id')
kategori_memnuniyet = df_kategori.groupby('product_category_name')['review_score'].mean().sort_values()

print("En düşük memnuniyet:")
print(kategori_memnuniyet.head(5))
print("\nEn yüksek memnuniyet:")
print(kategori_memnuniyet.tail(5))


In [ ]:
#Ürün Kategorisi vs. Memnuniyet grafiği
fig = px.bar(kategori_memnuniyet.reset_index(),
             x='review_score',
             y='product_category_name',
             title='Kategoriye Göre Ortalama Memnuniyet Puanı',
             labels={'review_score': 'Ortalama Puan', 'product_category_name': 'Kategori'},
             orientation='h',)

fig.show()

In [ ]:
#Ödeme Yöntemi vs. Memnuniyet: Ödeme yöntemi memnuniyetle ilişkili mi?
df_odeme = delivered_orders_copy.merge(payments[['order_id','payment_type']], on='order_id') \
                                 .merge(delivered_reviews[['order_id','review_score']], on='order_id')

odeme_memnuniyet = df_odeme.groupby('payment_type')['review_score'].mean().sort_values()
print(odeme_memnuniyet)

In [ ]:
#Ödeme Yöntemi vs. Memnuniyet Grafiği
fig = px.bar(odeme_memnuniyet.reset_index(),
             x='payment_type',
             y='review_score',
             title='Ödeme Yöntemine Göre Ortalama Memnuniyet Puanı',
             labels={'payment_type': 'Ödeme Yöntemi', 'review_score': 'Ortalama Puan'},)

fig.show()

In [ ]:
#Kargo Süresi vs. Memnuniyet: Daha hızlı teslimat, daha yüksek memnuniyet anlamına geliyor mu?
kargo_memnuniyet = df_analiz.groupby('review_score')['kargo_suresi'].mean()
print(kargo_memnuniyet)

In [ ]:
#Kargo Süresi vs. Memnuniyet Grafiği
fig = px.bar(kargo_memnuniyet.reset_index(),
             x='review_score',
             y='kargo_suresi',
             title='Memnuniyet Puanına Göre Ortalama Kargo Süresi',
             labels={'review_score': 'Memnuniyet Puanı', 'kargo_suresi': 'Ortalama Kargo Süresi (gün)'})

fig.show()

In [ ]:
#Teslimat Süresinin Müşteri Memnuniyetine Etkisi
from scipy import stats

df_analiz = delivered_orders_copy[['order_id','kargo_suresi','order_estimated_delivery_date','order_purchase_timestamp']] \
                .merge(delivered_items[['order_id','price','product_id']], on='order_id') \
                .merge(delivered_reviews[['order_id','review_score']], on='order_id')

df_analiz['tahmini_sure'] = (df_analiz['order_estimated_delivery_date'] -
                              df_analiz['order_purchase_timestamp']).dt.days

zamaninda = df_analiz[df_analiz['kargo_suresi'] <= df_analiz['tahmini_sure']]['review_score']
gec = df_analiz[df_analiz['kargo_suresi'] > df_analiz['tahmini_sure']]['review_score']

t_stat, p_value = stats.ttest_ind(zamaninda, gec)
print(f"Zamanında teslimat ortalama puan: {zamaninda.mean():.2f}")
print(f"Geç teslimat ortalama puan: {gec.mean():.2f}")
print(f"p-değeri: {p_value:.10f}")

In [ ]:
#Teslimat Süresinin Müşteri Memnuniyetine Etkisi Grafiği
fig = px.bar(x=['Zamanında Teslimat', 'Geç Teslimat'],
             y=[zamaninda.mean(), gec.mean()],
             title='Teslimat Süresine Göre Ortalama Memnuniyet',
             labels={'x': 'Teslimat Durumu', 'y': 'Ortalama Puan'})

fig.show()

In [ ]:
#Ürün Kategorisinin Sipariş Değerine Etkisi
kategoriler = df_analiz.merge(products[['product_id','product_category_name']], on='product_id')

gruplar = [grup['price'].values for _, grup in kategoriler.groupby('product_category_name')]

f_stat, p_value = stats.f_oneway(*gruplar)
print(f"F-istatistiği: {f_stat:.2f}")
print(f"p-değeri: {p_value:.10f}")

In [ ]:
#Ürün Kategorisinin Sipariş Değerine Etkisi Grafiği
kategori_fiyat = kategoriler.groupby('product_category_name')['price'].mean().sort_values().tail(10).reset_index()

fig = px.bar(kategori_fiyat,
             x='price',
             y='product_category_name',
             title='Kategoriye Göre Ortalama Sipariş Değeri (Top 10)',
             labels={'price': 'Ortalama Fiyat', 'product_category_name': 'Kategori'},
             orientation='h')
fig.show()

In [ ]:
#Ödeme Yönteminin Tamamlanma Oranına Etkisi
kredi = orders[orders['payment_type'] == 'credit_card'] if 'payment_type' in orders.columns else \
        orders.merge(payments[['order_id','payment_type']], on='order_id')

kredi_kart = orders.merge(payments[['order_id','payment_type']], on='order_id')

kredi = kredi_kart[kredi_kart['payment_type'] == 'credit_card']['order_status'].apply(lambda x: 1 if x == 'delivered' else 0)
boleto = kredi_kart[kredi_kart['payment_type'] == 'boleto']['order_status'].apply(lambda x: 1 if x == 'delivered' else 0)

t_stat, p_value = stats.ttest_ind(kredi, boleto)
print(f"Kredi kartı tamamlanma oranı: {kredi.mean()*100:.2f}%")
print(f"Boleto tamamlanma oranı: {boleto.mean()*100:.2f}%")
print(f"p-değeri: {p_value:.10f}")

In [ ]:
#Ödeme Yönteminin Tamamlanma Oranına Etkisi
fig = px.bar(x=['Kredi Kartı', 'Boleto'],
             y=[kredi.mean()*100, boleto.mean()*100],
             title='Ödeme Yöntemine Göre Tamamlanma Oranı',
             labels={'x': 'Ödeme Yöntemi', 'y': 'Tamamlanma Oranı (%)'})

fig.show()